# FableVisualizerV4 All-Pathways Runner (Notebook)

This notebook runs **all pathways** in a FABLE Excel workbook by driving Excel on Windows, then exports:
- recalculated workbooks per pathway
- CSVs for each output table (GHG, PRODUCTION, TRADE, JOBS, FOOD, LAND, WATER, N & P, BIODIVERSITY)
- CSVs for chart series source ranges
- combined tables and a run manifest

**Requirements**
- Windows + Microsoft Excel installed
- Python packages: `pandas`, `openpyxl`, `pywin32`

**How to run (Notebook)**
1) Run the code cell below.
2) Choose the workbook when prompted.
3) Choose an export folder (or cancel to use the default next to the workbook).
4) Watch the console output for progress.

**Where outputs are saved**
By default, outputs go to:
```
<workbook folder>\exports\all_pathways_run_YYYYMMDD_HHMMSS```
If you choose a custom export folder, the same `all_pathways_run_*` subfolder is created there.

**Dashboard (recommended)**
A Streamlit app (`FableVisualizerV4_Dashboard.py`) visualizes the outputs, including the **curated charts** which are direct mirrors of the ones seen in the FABLE calculator.
You can start it in two ways:
- Double-click `Launch_FableVisualizerV4_Launcher.bat` (no terminal)
- Or run the Tkinter launcher: `FableVisualizerV4_Launcher.py`

The launcher has three buttons (pick file, pick output folder, run dashboard), shows a **progress bar**, and displays the **current pathway** while exports run.


In [ ]:
# Run this cell to execute the runner
workbook = r""  # leave blank to pick a file
output_dir = None  # e.g., r"C:\\path\\to\\exports"
max_pathways = None  # e.g., 3 for a quick test
excel_visible = False  # set True to show Excel window (debug prompts)

# Load the runner from the .py file
from pathlib import Path
import sys
import importlib
import importlib.util

module_path = Path.cwd() / 'FableVisualizerV4_AllPathwaysRunner.py'
if not module_path.exists():
    raise FileNotFoundError(f'Module not found: {module_path}')
if str(module_path.parent) not in sys.path:
    sys.path.insert(0, str(module_path.parent))

if importlib.util.find_spec('win32com') is None:
    raise ModuleNotFoundError(
        'pywin32 is required in this Jupyter kernel. Install with `pip install pywin32` and restart the kernel.'
    )

import FableVisualizerV4_AllPathwaysRunner as runner
runner = importlib.reload(runner)
run_all_pathways = runner.run_all_pathways

# Pick workbook if not provided
if not workbook:
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        workbook = filedialog.askopenfilename(
            title='Select FABLE workbook',
            filetypes=[('Excel files', '*.xlsx *.xlsm'), ('All files', '*.*')],
        )
        root.destroy()
    except Exception as exc:
        raise RuntimeError(f'Failed to open file picker: {exc}')

# Ask for output folder (optional)
if output_dir is None:
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        output_dir = filedialog.askdirectory(
            title='Select export folder (cancel for default)'
        )
        root.destroy()
        if not output_dir:
            output_dir = None
    except Exception:
        output_dir = None

print(f'Selected workbook: {workbook}', flush=True)

if not workbook:
    print('No workbook selected. Stopping.', flush=True)
else:
    wb_path = Path(workbook)
    if not wb_path.exists():
        raise FileNotFoundError(f'Workbook not found: {wb_path}')

    output_root = (Path(output_dir) if output_dir else wb_path.parent / 'exports')
    print(f'Outputs will be saved to: {output_root}', flush=True)
    print('Starting run... this can take a few minutes.', flush=True)
    try:
        run_dir = run_all_pathways(
            workbook_path=wb_path,
            output_root=output_root,
            max_pathways=max_pathways,
            excel_visible=excel_visible,
        )
        print(f'Run complete. Outputs written to: {run_dir}', flush=True)
    except BaseException as exc:
        import traceback
        traceback.print_exc()
        raise


In [ ]:
# Plotly explorer for combined tables
# Edit table_name / metric / category settings below
from pathlib import Path
import pandas as pd
import plotly.express as px

# Try to use the run_dir from the previous cell; otherwise pick latest run in output_root
_run_dir = globals().get('run_dir', None)
if _run_dir is None:
    # fall back to output_root inferred from workbook (if available)
    _output_root = None
    if 'output_root' in globals():
        _output_root = Path(output_root)
    elif 'wb_path' in globals():
        _output_root = Path(wb_path).parent / 'exports'
    if _output_root is None or not _output_root.exists():
        raise FileNotFoundError('Could not determine output folder. Run the cell above first.')
    runs = [p for p in _output_root.glob('all_pathways_run_*') if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f'No run folders found in: {_output_root}')
    runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    _run_dir = runs[0]

run_dir = Path(_run_dir)
combined_dir = run_dir / 'combined_tables'
if not combined_dir.exists():
    raise FileNotFoundError(f'combined_tables not found in: {run_dir}')

tables = {p.stem.replace('__all_pathways', ''): p for p in combined_dir.glob('*.csv')}
print('Available tables:')
for name in sorted(tables):
    print(' -', name)

# ---- Choose what to plot ----
table_name = 'GHG__ResultsGHG'
metric = 'TotalCO2e'  # numeric column name in the table
category_col = None  # e.g., 'PROD_GROUP' or 'Product' (optional)
category_value = None  # e.g., 'CEREALS' (optional)
export_html = False

if table_name not in tables:
    raise ValueError(f'Table not found: {table_name}')

df = pd.read_csv(tables[table_name])
pathway_col = 'RunPathway' if 'RunPathway' in df.columns else '_RunPathway'

# detect year/x column (case-insensitive)
x_col = None
for c in df.columns:
    if str(c).strip().lower() == 'year':
        x_col = c
        break

if category_col and category_col in df.columns and category_value is not None:
    df = df[df[category_col] == category_value]

if metric not in df.columns:
    raise ValueError(f'Metric not found in {table_name}: {metric}')

if x_col:
    fig = px.line(
        df,
        x=x_col,
        y=metric,
        color=pathway_col,
        markers=True,
        title=f'{table_name} ? {metric}',
    )
else:
    # fallback to bar chart if no year column
    x_fallback = category_col or pathway_col
    fig = px.bar(
        df,
        x=x_fallback,
        y=metric,
        color=pathway_col,
        title=f'{table_name} ? {metric}',
    )

fig.show()

if export_html:
    out_dir = run_dir / 'plotly_charts'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{table_name}__{metric}.html'
    fig.write_html(out_path)
    print(f'Saved: {out_path}')


In [ ]:
# Scenario deviation analysis: single CSV summary + selective high-deviation charts
from pathlib import Path
import re
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

# ---- Settings ----
analysis_dir_name = 'scenario_deviation_analysis'
focus_tables = None  # e.g., ['GHG__ResultsGHG']
focus_metrics = None  # e.g., ['TotalCO2e', 'Area']
baseline_pathway = None  # e.g., 'Baseline'; None = alphabetical first in each comparison
default_abs_deviation_threshold = 0.0
default_pct_deviation_threshold = 15.0  # significant if spread percent >= this threshold
metric_thresholds = {
    # 'TotalCO2e': {'abs': 1000, 'pct': 10},
}
max_graphs = 12
export_graphs_html = True

# ---- Resolve run directory ----
_run_dir = globals().get('run_dir', None)
if _run_dir is None:
    _output_root = None
    if 'output_root' in globals():
        _output_root = Path(output_root)
    elif 'wb_path' in globals():
        _output_root = Path(wb_path).parent / 'exports'
    if _output_root is None or not _output_root.exists():
        raise FileNotFoundError('Could not determine output folder. Run the runner cell first.')
    runs = [p for p in _output_root.glob('all_pathways_run_*') if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f'No run folders found in: {_output_root}')
    runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    _run_dir = runs[0]

run_dir = Path(_run_dir)
combined_dir = run_dir / 'combined_tables'
if not combined_dir.exists():
    raise FileNotFoundError(f'combined_tables not found in: {run_dir}')

analysis_dir = run_dir / analysis_dir_name
analysis_dir.mkdir(parents=True, exist_ok=True)

def safe_name(text):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', str(text)).strip('._') or 'unnamed'

def as_group_text(row, dims):
    if not dims:
        return '(all rows)'
    parts = []
    for c in dims:
        v = row.get(c, None)
        if pd.isna(v):
            v = '<NA>'
        parts.append(f'{c}={v}')
    return ' | '.join(parts)

def thresholds_for(metric):
    spec = metric_thresholds.get(metric, {})
    abs_thr = float(spec.get('abs', default_abs_deviation_threshold))
    pct_thr = float(spec.get('pct', default_pct_deviation_threshold))
    return abs_thr, pct_thr

table_files = sorted(combined_dir.glob('*.csv'))
if focus_tables:
    wanted = set(focus_tables)
    table_files = [p for p in table_files if p.stem.replace('__all_pathways', '') in wanted]

if not table_files:
    raise FileNotFoundError('No combined table CSV files matched the current filters.')

summary_frames = []
plot_candidates = []

for path in table_files:
    table_name = path.stem.replace('__all_pathways', '')
    df = pd.read_csv(path)
    if df.empty:
        continue

    pathway_col = 'RunPathway' if 'RunPathway' in df.columns else ('_RunPathway' if '_RunPathway' in df.columns else None)
    if pathway_col is None or df[pathway_col].nunique(dropna=True) < 2:
        continue

    # Treat "Year" as a dimension even when numeric.
    year_cols = [c for c in df.columns if str(c).strip().lower() == 'year']
    dimension_cols = [c for c in df.columns if c != pathway_col and (df[c].dtype == object or c in year_cols)]

    numeric_metrics = []
    for c in df.columns:
        if c == pathway_col or c in dimension_cols:
            continue
        s = pd.to_numeric(df[c], errors='coerce')
        if s.notna().any():
            df[c] = s
            numeric_metrics.append(c)

    if focus_metrics:
        allowed = set(focus_metrics)
        numeric_metrics = [m for m in numeric_metrics if m in allowed]

    if not numeric_metrics:
        continue

    for metric in numeric_metrics:
        cols = dimension_cols + [pathway_col, metric]
        work = df[cols].copy()
        work[metric] = pd.to_numeric(work[metric], errors='coerce')
        work = work.dropna(subset=[metric])
        if work.empty:
            continue

        agg = work.groupby(dimension_cols + [pathway_col], dropna=False, as_index=False)[metric].sum()

        if dimension_cols:
            pivot = agg.pivot_table(
                index=dimension_cols,
                columns=pathway_col,
                values=metric,
                aggfunc='sum',
            ).reset_index()
        else:
            totals = agg.groupby(pathway_col, as_index=False)[metric].sum()
            if totals.empty:
                continue
            pivot = pd.DataFrame([totals.set_index(pathway_col)[metric].to_dict()])

        pivot.columns.name = None
        pathway_names = [c for c in pivot.columns if c not in dimension_cols]
        if len(pathway_names) < 2:
            continue

        pathway_names = sorted(pathway_names, key=lambda x: str(x))
        vals = pivot[pathway_names]

        baseline = baseline_pathway if baseline_pathway in pathway_names else pathway_names[0]
        baseline_values = pd.to_numeric(pivot[baseline], errors='coerce')

        max_values = vals.max(axis=1)
        min_values = vals.min(axis=1)
        spread_abs = max_values - min_values
        mean_abs = vals.abs().mean(axis=1)
        spread_pct = np.where(mean_abs > 0, (spread_abs / mean_abs) * 100.0, np.nan)
        spread_pct_series = pd.Series(spread_pct, index=pivot.index)

        abs_thr, pct_thr = thresholds_for(metric)
        significant = (spread_abs.fillna(0) >= abs_thr) & (spread_pct_series.fillna(0) >= pct_thr)

        out = pivot[dimension_cols].copy() if dimension_cols else pd.DataFrame(index=pivot.index)
        out.insert(0, 'Table', table_name)
        out.insert(1, 'Metric', metric)
        out['Group'] = out.apply(lambda r: as_group_text(r, dimension_cols), axis=1) if dimension_cols else '(all rows)'
        out['BaselinePathway'] = baseline
        out['BaselineValue'] = baseline_values
        out['MaxPathway'] = vals.idxmax(axis=1)
        out['MaxValue'] = max_values
        out['MinPathway'] = vals.idxmin(axis=1)
        out['MinValue'] = min_values
        out['SpreadAbs'] = spread_abs
        out['SpreadPct'] = spread_pct_series
        out['AbsDeviationThreshold'] = abs_thr
        out['PctDeviationThreshold'] = pct_thr
        out['SignificantDeviation'] = significant

        for p_name in pathway_names:
            col = safe_name(p_name)
            out[f'Value__{col}'] = pd.to_numeric(pivot[p_name], errors='coerce')
            out[f'DiffVsBaseline__{col}'] = out[f'Value__{col}'] - baseline_values

        summary_frames.append(out)

        # Chart candidate: include only metrics with significant spread.
        if bool(pd.Series(significant).any()):
            year_col = year_cols[0] if year_cols else None
            plot_score = float(spread_pct_series.fillna(0).max())
            if year_col and year_col in agg.columns:
                plot_df = agg.groupby([year_col, pathway_col], as_index=False)[metric].sum()
                if plot_df[year_col].nunique() > 1:
                    plot_candidates.append(
                        {
                            'table': table_name,
                            'metric': metric,
                            'score': plot_score,
                            'kind': 'line',
                            'x': year_col,
                            'pathway_col': pathway_col,
                            'data': plot_df,
                        }
                    )
            else:
                plot_df = agg.groupby([pathway_col], as_index=False)[metric].sum()
                plot_candidates.append(
                    {
                        'table': table_name,
                        'metric': metric,
                        'score': plot_score,
                        'kind': 'bar',
                        'x': pathway_col,
                        'pathway_col': pathway_col,
                        'data': plot_df,
                    }
                )

if not summary_frames:
    raise RuntimeError('No comparable multi-pathway numeric data found in combined_tables with the current filters.')

summary_df = pd.concat(summary_frames, ignore_index=True)
summary_df = summary_df.sort_values(['SignificantDeviation', 'SpreadPct', 'SpreadAbs'], ascending=[False, False, False])

summary_csv = analysis_dir / 'scenario_deviation_summary.csv'
summary_df.to_csv(summary_csv, index=False)

sig_df = summary_df[summary_df['SignificantDeviation'] == True]
print(f'Saved analysis CSV: {summary_csv}')
print(f'Total comparison rows: {len(summary_df)}')
print(f'Significant deviation rows: {len(sig_df)}')

if not sig_df.empty:
    preview_cols = ['Table', 'Metric', 'Group', 'SpreadAbs', 'SpreadPct', 'MaxPathway', 'MinPathway']
    display(sig_df[preview_cols].head(20))

# ---- Selective graph export (significant deviations only) ----
plot_dir = analysis_dir / 'significant_deviation_charts'
plot_dir.mkdir(parents=True, exist_ok=True)

seen = set()
selected = []
for c in sorted(plot_candidates, key=lambda d: d['score'], reverse=True):
    key = (c['table'], c['metric'], c['kind'])
    if key in seen:
        continue
    seen.add(key)
    selected.append(c)
    if max_graphs is not None and len(selected) >= max_graphs:
        break

for i, c in enumerate(selected, start=1):
    title = f"{c['table']} | {c['metric']} (high deviation)"
    df_plot = c['data'].copy()

    if c['kind'] == 'line':
        fig = px.line(
            df_plot,
            x=c['x'],
            y=c['metric'],
            color=c['pathway_col'],
            markers=True,
            title=title,
        )
    else:
        fig = px.bar(
            df_plot,
            x=c['x'],
            y=c['metric'],
            color=c['pathway_col'] if c['x'] != c['pathway_col'] else None,
            barmode='group',
            title=title,
        )

    fig.show()
    if export_graphs_html:
        out_name = f"{i:02d}__{safe_name(c['table'])}__{safe_name(c['metric'])}.html"
        fig.write_html(plot_dir / out_name)

if selected:
    print(f'Selective charts generated: {len(selected)}')
    print(f'Chart folder: {plot_dir}')
else:
    print('No charts met the significant deviation criteria with the current thresholds.')


In [ ]:
# Batch export Plotly charts for ALL combined tables
from pathlib import Path
import re
import pandas as pd
import plotly.express as px

# ---- Settings ----
export_html = True
export_dir_name = 'plotly_batch'
only_tables = None  # e.g., ['GHG__ResultsGHG', 'PRODUCTION__ResultsProd'] or None for all
exclude_tables = []
max_charts = None  # set an int to cap chart count

# For tables with both Year + categories, optionally split by a category column
category_split = {
    'FOOD__Results_Diets': 'PROD_GROUP',
    'PRODUCTION__ResultsProd': 'Product',
    'PRODUCTION__TotalResultsProd': 'Product',
}
max_category_values = 10  # limit per table when splitting

# ---- Resolve run directory ----
_run_dir = globals().get('run_dir', None)
if _run_dir is None:
    _output_root = None
    if 'output_root' in globals():
        _output_root = Path(output_root)
    elif 'wb_path' in globals():
        _output_root = Path(wb_path).parent / 'exports'
    if _output_root is None or not _output_root.exists():
        raise FileNotFoundError('Could not determine output folder. Run the cell above first.')
    runs = [p for p in _output_root.glob('all_pathways_run_*') if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f'No run folders found in: {_output_root}')
    runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    _run_dir = runs[0]

run_dir = Path(_run_dir)
combined_dir = run_dir / 'combined_tables'
if not combined_dir.exists():
    raise FileNotFoundError(f'combined_tables not found in: {run_dir}')

def safe_name(text: str) -> str:
    cleaned = re.sub(r'[^A-Za-z0-9._-]+', '_', str(text)).strip('._')
    return cleaned or 'unnamed'

tables = {p.stem.replace('__all_pathways', ''): p for p in combined_dir.glob('*.csv')}
if only_tables:
    tables = {k: v for k, v in tables.items() if k in set(only_tables)}
if exclude_tables:
    tables = {k: v for k, v in tables.items() if k not in set(exclude_tables)}

export_dir = run_dir / export_dir_name
export_dir.mkdir(parents=True, exist_ok=True)

chart_count = 0
for table_name, path in sorted(tables.items()):
    df = pd.read_csv(path)
    if df.empty:
        continue

    # Identify key columns
    path_col = 'RunPathway' if 'RunPathway' in df.columns else ('_RunPathway' if '_RunPathway' in df.columns else None)
    year_col = None
    for c in df.columns:
        if str(c).strip().lower() == 'year':
            year_col = c
            break

    # categorical columns (object dtype)
    cat_cols = [c for c in df.columns if c not in {path_col, year_col} and df[c].dtype == object]

    # numeric columns
    numeric_cols = []
    for c in df.columns:
        if c in {path_col, year_col} or c in cat_cols:
            continue
        series = pd.to_numeric(df[c], errors='coerce')
        if series.notna().any():
            df[c] = series
            numeric_cols.append(c)

    if not numeric_cols:
        continue

    out_table_dir = export_dir / safe_name(table_name)
    out_table_dir.mkdir(parents=True, exist_ok=True)

    if year_col:
        if table_name in category_split and category_split[table_name] in df.columns:
            cat_col = category_split[table_name]
            values = [v for v in df[cat_col].dropna().unique().tolist()]
            if max_category_values:
                values = values[:max_category_values]
            for val in values:
                sub = df[df[cat_col] == val]
                if path_col:
                    sub = sub.groupby([path_col, year_col], as_index=False)[numeric_cols].sum()
                for metric in numeric_cols:
                    if max_charts is not None and chart_count >= max_charts:
                        break
                    fig = px.line(
                        sub,
                        x=year_col,
                        y=metric,
                        color=path_col,
                        markers=True,
                        title=f'{table_name} ? {metric} ({cat_col}={val})',
                    )
                    if export_html:
                        out_name = f'{safe_name(metric)}__{safe_name(cat_col)}_{safe_name(val)}.html'
                        fig.write_html(out_table_dir / out_name)
                    chart_count += 1
                if max_charts is not None and chart_count >= max_charts:
                    break
        else:
            # Aggregate across categories to avoid duplicate year rows
            if path_col:
                plot_df = df.groupby([path_col, year_col], as_index=False)[numeric_cols].sum()
            else:
                plot_df = df.groupby([year_col], as_index=False)[numeric_cols].sum()

            for metric in numeric_cols:
                if max_charts is not None and chart_count >= max_charts:
                    break
                fig = px.line(
                    plot_df,
                    x=year_col,
                    y=metric,
                    color=path_col if path_col else None,
                    markers=True,
                    title=f'{table_name} ? {metric}',
                )
                if export_html:
                    out_name = f'{safe_name(metric)}.html'
                    fig.write_html(out_table_dir / out_name)
                chart_count += 1
            if max_charts is not None and chart_count >= max_charts:
                break
    else:
        # No year column: make grouped bar charts
        x_col = None
        if cat_cols:
            x_col = cat_cols[0]
        else:
            x_col = path_col

        plot_df = df
        if path_col and x_col and path_col != x_col:
            plot_df = plot_df.groupby([path_col, x_col], as_index=False)[numeric_cols].sum()
        elif x_col:
            plot_df = plot_df.groupby([x_col], as_index=False)[numeric_cols].sum()

        for metric in numeric_cols:
            if max_charts is not None and chart_count >= max_charts:
                break
            fig = px.bar(
                plot_df,
                x=x_col,
                y=metric,
                color=path_col if path_col and path_col != x_col else None,
                barmode='group',
                title=f'{table_name} ? {metric}',
            )
            if export_html:
                out_name = f'{safe_name(metric)}.html'
                fig.write_html(out_table_dir / out_name)
            chart_count += 1
        if max_charts is not None and chart_count >= max_charts:
            break

print(f'Batch export complete. Charts saved to: {export_dir}')
